# Notebook 12 — Multiplication in GF(2⁸)

**Volume 1: Foundations, Finite Fields, and AES**

## Sections
1. Introduction and setup
2. Byte-polynomial conversion utilities
3. xtime: the byte-level shortcut
4. Verifying xtime with polynomial view
5. Multiply by 0x02 and 0x03
6. General GF(2⁸) multiplication via xtime chain
7. Worked example: 0x57 × 0x13
8. Verification via full polynomial multiply + reduce
9. Interactive multiplier: gf_mul_verbose(a, b)
10. Summary and bridge to Module 13

## Section 1 — Introduction

Module 11 showed us how to reduce polynomials modulo the AES irreducible polynomial

    m(x) = x^8 + x^4 + x^3 + x + 1   (hex: 0x11B)

This module combines that reduction with polynomial multiplication to produce **full GF(2⁸) multiplication** — the byte-level arithmetic at the core of AES MixColumns.

The recipe has three stages:
1. **Convert** each byte to a polynomial over GF(2).
2. **Multiply** the two polynomials (XOR-based, no carries).
3. **Reduce** the product modulo m(x) to get a byte-sized result.

The `xtime` function is a fast special case for multiplying by x (= 0x02). Building a chain of xtime calls lets us multiply any two bytes efficiently.

In [ ]:
# Section 2 — Byte-polynomial conversion utilities
# (Re-declared from Module 11 so this notebook is self-contained.)
#
# A polynomial over GF(2) is stored as an integer bitmask.
# Bit d == coefficient of x^d  (LSB = constant term).

def poly_to_str(v, max_deg=14):
    """Convert bitmask to human-readable polynomial string."""
    if v == 0:
        return '0'
    terms = []
    for d in range(max_deg, -1, -1):
        if (v >> d) & 1:
            if   d == 0: terms.append('1')
            elif d == 1: terms.append('x')
            else:        terms.append(f'x^{d}')
    return ' + '.join(terms)

def poly_from_coeffs(*coeffs):
    """Build bitmask from list of exponents with coefficient 1.
    e.g. poly_from_coeffs(8,4,3,1,0) -> 0x11B
    """
    v = 0
    for c in coeffs:
        v |= (1 << c)
    return v

AES_MOD = poly_from_coeffs(8, 4, 3, 1, 0)   # 0x11B
FOLD_8  = AES_MOD ^ (1 << 8)                 # 0x1B  (lower part of modulus)

print(f'AES modulus : {poly_to_str(AES_MOD)}  = 0x{AES_MOD:03X}')
print(f'Folded x^8  : {poly_to_str(FOLD_8)}  = 0x{FOLD_8:02X}')

# Quick demo: byte 0x57
b57 = 0x57
print(f'\n0x{b57:02X} = {b57:08b}b = {poly_to_str(b57)}')

In [ ]:
# Section 3 — xtime: the byte-level shortcut
#
# xtime(a) = a * x in GF(2^8) = a * 0x02
#
# Algorithm:
#   1. Left-shift a by 1, keep only 8 bits.
#   2. If the original top bit (bit 7) was 1, XOR with 0x1B.

def xtime(a):
    """Multiply byte a by x (= 0x02) in GF(2^8)."""
    shifted = (a << 1) & 0xFF
    return shifted ^ 0x1B if (a & 0x80) else shifted

# Demo: xtime(0x57) — top bit is 0, no reduction
a = 0x57
r = xtime(a)
print(f'xtime(0x{a:02X}) = 0x{r:02X}')
print(f'  Binary: {a:08b} -> shift -> {r:08b} (top bit was 0, no XOR)')

print()

# Demo: xtime(0x83) — top bit is 1, reduction needed
a = 0x83
r = xtime(a)
shifted = (a << 1) & 0xFF
print(f'xtime(0x{a:02X}) = 0x{r:02X}')
print(f'  Binary: {a:08b} -> shift low byte -> {shifted:08b} = 0x{shifted:02X}')
print(f'  Top bit was 1, so XOR 0x1B: 0x{shifted:02X} ^ 0x1B = 0x{r:02X}')

In [ ]:
# Section 4 — Verifying xtime with the polynomial view
#
# We confirm the byte-level shortcut matches the polynomial calculation:
#   multiply a(x) by x, then reduce mod m(x).

def poly_degree(v):
    return -1 if v == 0 else v.bit_length() - 1

def poly_mul_gf2(a, b):
    """Multiply two polynomials over GF(2) without reduction."""
    result = 0
    while a:
        if a & 1: result ^= b
        b <<= 1
        a >>= 1
    return result

def poly_reduce(val, modulus=AES_MOD):
    """Reduce val mod modulus over GF(2)."""
    deg_m = poly_degree(modulus)
    cur = val
    for d in range(poly_degree(cur), deg_m - 1, -1):
        if (cur >> d) & 1:
            cur ^= (modulus << (d - deg_m))
    return cur & ((1 << deg_m) - 1)

print('Verifying xtime via polynomial multiplication + reduction:')
print()
for a in [0x57, 0x83, 0xAE, 0x01, 0xFF]:
    # Polynomial method: multiply by x (= 0b10 = 2), then reduce
    product_poly = poly_mul_gf2(a, 0b10)      # a(x) * x, unreduced
    reduced_poly = poly_reduce(product_poly)   # mod m(x)
    # Byte-level method
    byte_result  = xtime(a)
    match = '✓' if reduced_poly == byte_result else '✗ MISMATCH'
    print(f'xtime(0x{a:02X}): poly={0x{reduced_poly:02X}}, byte=0x{byte_result:02X}  {match}')

In [ ]:
# Section 5 — Multiply by 0x02 and 0x03
#
# Multiplying by 0x02 IS xtime.
# Multiplying by 0x03 = x + 1 means:  a*(x+1) = a*x + a = xtime(a) XOR a

def mul_02(a):
    """Multiply byte a by 0x02 in GF(2^8) — same as xtime."""
    return xtime(a)

def mul_03(a):
    """Multiply byte a by 0x03 in GF(2^8)."""
    return xtime(a) ^ a

test_bytes = [0x57, 0x83, 0xAE, 0x01, 0x00]

print('Multiplication by 0x02 and 0x03:')
print(f'{"Byte":>6}  {"× 0x02":>8}  {"× 0x03":>8}')
print('-' * 32)
for a in test_bytes:
    print(f'0x{a:02X}    0x{mul_02(a):02X}      0x{mul_03(a):02X}')

print()
print('Note: 0x57 × 0x03 = 0xAE XOR 0x57 =', hex(0xAE ^ 0x57))

In [ ]:
# Section 6 — General GF(2^8) multiplication via xtime chain
#
# To multiply a × b, we decompose b into powers of x and use repeated xtime.
# For each bit i set in b, we need x^i * a.
# x^0 * a = a
# x^1 * a = xtime(a)
# x^2 * a = xtime(xtime(a))
# ... and so on.
# The result is the XOR of all selected chain values.

def gf_mul(a, b):
    """Multiply bytes a and b in GF(2^8) using the xtime chain."""
    result = 0
    aa = a
    for i in range(8):
        if (b >> i) & 1:
            result ^= aa
        aa = xtime(aa)
    return result

# Quick spot checks
print('GF(2^8) multiplication spot checks:')
checks = [
    (0x57, 0x01, 0x57),    # a * 1 = a
    (0x57, 0x02, 0xAE),    # a * 0x02 = xtime(a)
    (0x57, 0x03, 0xF9),    # a * 0x03 = xtime(a) XOR a
    (0x57, 0x13, 0xFE),    # worked example from tutorial
    (0x83, 0x02, 0x1D),    # xtime(0x83) with reduction
]
for a, b, expected in checks:
    result = gf_mul(a, b)
    status = '✓' if result == expected else f'✗ (expected 0x{expected:02X})'
    print(f'  0x{a:02X} × 0x{b:02X} = 0x{result:02X}  {status}')

In [ ]:
# Section 7 — Worked example: 0x57 × 0x13
#
# 0x13 = 0001 0011 = x^4 + x + 1
# So we need x^4*a XOR x^1*a XOR x^0*a  where a = 0x57.

a = 0x57
b = 0x13

print(f'Computing 0x{a:02X} × 0x{b:02X} in GF(2^8)')
print(f'0x{b:02X} = {b:08b}b = {poly_to_str(b)}')
print()

# Build the xtime chain for a
print('xtime chain for A = 0x57:')
chain = [a]
for i in range(1, 8):
    chain.append(xtime(chain[-1]))
for i, v in enumerate(chain):
    print(f'  x^{i} * A = 0x{v:02X}')

print()
# Select the entries corresponding to set bits in b
selected = [(i, chain[i]) for i in range(8) if (b >> i) & 1]
print('B selects:')
for i, v in selected:
    print(f'  x^{i} * A = 0x{v:02X}')

print()
result = 0
for i, v in selected:
    result ^= v
xor_str = ' XOR '.join(f'0x{v:02X}' for _, v in selected)
print(f'Result = {xor_str} = 0x{result:02X}')

In [ ]:
# Section 8 — Verification via full polynomial multiply + reduce
#
# We verify 0x57 × 0x13 by expanding the full polynomial product
# and then reducing mod m(x).

a = 0x57   # x^6 + x^4 + x^2 + x + 1
b = 0x13   # x^4 + x + 1

print(f'0x{a:02X} = {poly_to_str(a)}')
print(f'0x{b:02X} = {poly_to_str(b)}')
print()

# Unreduced product
unreduced = poly_mul_gf2(a, b)
print(f'Unreduced product: {poly_to_str(unreduced)}')
print(f'  (degree {poly_degree(unreduced)}, hex 0x{unreduced:04X})')
print()

# Reduce mod AES modulus
reduced = poly_reduce(unreduced)
print(f'After reduction mod m(x): {poly_to_str(reduced)} = 0x{reduced:02X}')
print()

# Cross-check with gf_mul
fast = gf_mul(a, b)
print(f'gf_mul result: 0x{fast:02X}')
print(f'Both methods agree: {reduced == fast}')

In [ ]:
# Section 9 — Interactive multiplier: gf_mul_verbose(a, b)
#
# Call gf_mul_verbose(a, b) to see every step of the multiplication.

def gf_mul_verbose(a, b):
    """
    Multiply bytes a and b in GF(2^8), printing every step.

    Parameters
    ----------
    a, b : int  — byte values (0..255)

    Returns
    -------
    int — the product a * b in GF(2^8)
    """
    print(f'Computing 0x{a:02X} × 0x{b:02X} in GF(2^8)')
    print(f'  A = 0x{a:02X} = {poly_to_str(a)}')
    print(f'  B = 0x{b:02X} = {b:08b}b = {poly_to_str(b)}')
    print()

    # Build and print xtime chain
    print('  xtime chain for A:')
    chain = [a]
    for i in range(1, 8):
        chain.append(xtime(chain[-1]))
    for i, v in enumerate(chain):
        selected = '  <-- B has bit ' + str(i) if (b >> i) & 1 else ''
        print(f'    x^{i} * A = 0x{v:02X}{selected}')

    # Select and XOR
    selected_vals = [(i, chain[i]) for i in range(8) if (b >> i) & 1]
    print()
    if not selected_vals:
        print('  B = 0 → result = 0x00')
        return 0
    print('  XOR selected values:')
    result = 0
    running = []
    for i, v in selected_vals:
        result ^= v
        running.append(f'0x{v:02X}')
        print(f'    {" XOR ".join(running)} = 0x{result:02X}')

    print()
    print(f'  Result: 0x{a:02X} × 0x{b:02X} = 0x{result:02X}')
    return result

# Try the main worked example
print('=' * 50)
gf_mul_verbose(0x57, 0x13)
print()
print('=' * 50)
gf_mul_verbose(0x57, 0x0B)

## Section 10 — Summary and bridge to Module 13

### Key results from this notebook

| Concept | Formula / Detail |
|---|---|
| GF(2⁸) element | A byte = polynomial with coefficients 0 or 1 |
| Addition | XOR (`^`) |
| xtime | `xtime(a)` = left shift, XOR 0x1B if top bit was 1 |
| × 0x02 | Same as `xtime(a)` |
| × 0x03 | `xtime(a) ^ a` |
| General multiply | Build xtime chain; XOR entries selected by bits of b |
| AES modulus | 0x11B = `x^8 + x^4 + x^3 + x + 1`; fold constant = 0x1B |
| Worked example | 0x57 × 0x13 = 0xFE |

### Key functions
```python
xtime(a)          # multiply byte a by 0x02 in GF(2^8)
gf_mul(a, b)      # multiply any two bytes in GF(2^8)
gf_mul_verbose(a, b)  # same, with step-by-step output
```

### Bridge to Module 13: Multiplicative Inverses in GF(2⁸)

Every **nonzero** byte in GF(2⁸) has a multiplicative inverse: a unique byte `b` such that

    gf_mul(a, b) == 0x01

Module 13 explores how to compute these inverses — via the Extended Euclidean Algorithm over polynomials — and why this is the mathematical heart of the AES S-box.